# 🔷 @contextmanager and Decorators — Complete Reference Guide

How Python's `@contextmanager` turns a plain generator into a full context
manager — and how to compose one inside a decorator for reusable resource timing.

---

## Table of Contents

1. [What Is a Context Manager?](#1-what-is-a-context-manager)
2. [@contextmanager — The Generator Shortcut](#2-contextmanager--the-generator-shortcut)
3. [Execution Flow — Step by Step](#3-execution-flow--step-by-step)
4. [Why Yield a Mutable Container?](#4-why-yield-a-mutable-container)
5. [Decorators Recap — Two-Layer vs Three-Layer](#5-decorators-recap--two-layer-vs-three-layer)
6. [Wrapping a Context Manager in a Decorator](#6-wrapping-a-context-manager-in-a-decorator)
7. [Common Patterns and Real-World Usage](#7-common-patterns-and-real-world-usage)
8. [Gotchas and Pitfalls](#8-gotchas-and-pitfalls)

---

## 1. What Is a Context Manager?

A **context manager** is any object that implements two dunder methods:

| Method | When it runs | Role |
|---|---|---|
| `__enter__(self)` | When the `with` block **starts** | Set up; its return value → `as X` |
| `__exit__(self, exc_type, exc_val, exc_tb)` | When the block **ends** — always | Tear down; return `False` to propagate exceptions |

The `with` statement **guarantees** `__exit__` is called — even if an exception
fires inside the block. That guarantee makes it the right tool for any resource
that must be released: files, database connections, locks, timers.

**Mental model — The Bouncer Pattern:**

```
with open("file.txt") as f:    # ① Bouncer opens the door  (__enter__)
    data = f.read()            # ② You are inside the club  (your code)
                               # ③ Bouncer closes the door  (__exit__)
                               #    Door closes EVEN IF you cause trouble
```

**Why it matters:**

```python
# Without `with` — manual, fragile:
f = open("file.txt")
data = f.read()    # ← if this raises, f.close() never runs → resource leak
f.close()

# With `with` — automatic, guaranteed:
with open("file.txt") as f:
    data = f.read()    # ← even if this raises, __exit__ still runs
```

In [1]:
# =============================================================================
# CLASS-BASED CONTEXT MANAGER (THE VERBOSE WAY)
# =============================================================================
from __future__ import annotations   # PEP 604 unions + forward refs for the whole session

import time


class ManualTimer:
    """Times a code block using the explicit __enter__ / __exit__ protocol.

    This is the class-based (verbose) way to write a context manager.
    Section 2 shows how @contextmanager cuts this to ~5 lines.

    Parameters
    ----------
    label : str
        Human-readable name shown in diagnostic output.

    Examples
    --------
    >>> with ManualTimer("demo") as t:
    ...     _ = sum(range(100_000))
    >>> t.elapsed   # float, populated after the block
    """

    def __init__(self, label: str) -> None:
        self.label = label
        self.elapsed: float = 0.0           # populated by __exit__

    def __enter__(self) -> ManualTimer:
        """Called when the `with` block starts; its return value → `as t`.

        Returns
        -------
        ManualTimer
            `self` — so the caller can do `as t` and inspect attributes.
        """
        self._start = time.perf_counter()   # high-resolution monotonic clock
        print(f"  __enter__ : '{self.label}' started")
        return self                          # ← assigned to `t` in `with … as t`

    def __exit__(self, exc_type, exc_val, exc_tb) -> bool:
        """Called when the `with` block ends — always, even on exceptions.

        Parameters
        ----------
        exc_type : type | None
            Exception class if an error occurred, else None.
        exc_val : BaseException | None
            Exception instance if an error occurred, else None.
        exc_tb : TracebackType | None
            Traceback if an error occurred, else None.

        Returns
        -------
        bool
            False → let exceptions propagate (almost always right).
            True  → swallow the exception — dangerous, hides bugs.
        """
        self.elapsed = time.perf_counter() - self._start
        print(f"  __exit__  : '{self.label}' stopped — {self.elapsed:.6f}s")
        return False   # do NOT swallow exceptions

In [2]:
# =============================================================================
# USING ManualTimer
# =============================================================================

with ManualTimer("demo") as t:
    # The `with` statement calls __enter__, runs this block, then calls __exit__
    total = sum(range(100_000))

# __exit__ has already run; t.elapsed is now populated
print(f"sum     = {total}")
print(f"elapsed = {t.elapsed:.6f}s")

  __enter__ : 'demo' started
  __exit__  : 'demo' stopped — 0.000845s
sum     = 4999950000
elapsed = 0.000845s


In [3]:
# =============================================================================
# WHAT PYTHON DOES BEHIND THE SCENES
# =============================================================================

# `with X as t:` is roughly syntactic sugar for:
#
#   mgr = ManualTimer("demo")
#   t   = mgr.__enter__()       # ← __enter__ return value → `t`
#   try:
#       body                    # ← your code runs here
#   except:
#       if not mgr.__exit__(*sys.exc_info()):
#           raise               # ← re-raise if __exit__ returns False
#   else:
#       mgr.__exit__(None, None, None)   # ← happy path cleanup
#
# Python guarantees __exit__ is called in BOTH branches.
# That is the entire value of the with statement.

print(f"t.__class__.__name__ = {t.__class__.__name__!r}")   # still accessible after block

t.__class__.__name__ = 'ManualTimer'


---

## 2. `@contextmanager` — The Generator Shortcut

Writing `__enter__` and `__exit__` is verbose. `contextlib.contextmanager`
converts a **generator function** with exactly one `yield` into a full
context manager — no class needed.

**Structure:**

```python
from contextlib import contextmanager

@contextmanager
def my_context(arg):
    # ── everything before yield ──  runs as __enter__
    setup(arg)

    yield value     # value → `as X`; generator PAUSES here

    # ── everything after yield ──   runs as __exit__
    cleanup()
```

**How `@contextmanager` works under the hood:**

It wraps your generator in a helper class that calls `next()` on it twice —
once to run to `yield` (acting as `__enter__`) and once to resume to the end
(acting as `__exit__`). If an exception fires in the body, it is *thrown into*
the generator at the yield point via `generator.throw()`.

**Class-based vs `@contextmanager` — same behaviour, a fraction of the code:**

```
Class-based (13 lines)                 @contextmanager (5 lines)
───────────────────────────────────    ───────────────────────────────────
class Timer:                           @contextmanager
    def __init__(self, label):         def timer(label):
        self.label = label                 container: dict = {}
    def __enter__(self):                   start = perf_counter()
        self._start = perf_counter()       yield container
        return self                        container["elapsed"] = (
    def __exit__(self, *a):                    perf_counter() - start)
        self.elapsed = ...
        return False
```

In [4]:
# =============================================================================
# @contextmanager — DEFINITION
# =============================================================================
from contextlib import contextmanager
from typing import Any, Generator


@contextmanager
def simple_timer(label: str) -> Generator[dict[str, Any], None, None]:
    """Time a code block; store the result in a mutable dict.

    Yields a dict so the caller can read ``t["elapsed"]`` *after* the
    block ends. (Why a dict instead of a float? Section 4 explains.)

    Parameters
    ----------
    label : str
        Name shown in diagnostic output.

    Yields
    ------
    dict[str, Any]
        Empty on entry; ``"elapsed"`` (float, seconds) added after the block.

    Examples
    --------
    >>> with simple_timer("load") as t:
    ...     data = load_something()
    >>> t["elapsed"]   # seconds
    """
    container: dict[str, Any] = {}     # mutable — caller and generator share this object

    # ── BEFORE yield ──  (__enter__ equivalent)
    start = time.perf_counter()
    print(f"  [before yield] '{label}' started")

    yield container                    # generator PAUSES here; `as t` → this dict

    # ── AFTER yield ──   (__exit__ equivalent) — resumes when the block ends
    elapsed = time.perf_counter() - start
    container["elapsed"] = elapsed     # writes into the dict the caller already holds
    print(f"  [after yield]  '{label}' finished — {elapsed:.6f}s")

In [5]:
# =============================================================================
# USING simple_timer
# =============================================================================

# The three prints — [before yield], user code, [after yield] — fire in order,
# proving that yield literally pauses the generator between setup and cleanup.
with simple_timer("sum_demo") as t:
    print(f"  [inside with]  user code running...")   # fires BETWEEN before and after
    total = sum(range(100_000))

print(f"t['elapsed'] = {t['elapsed']:.6f}s  |  sum = {total}")

  [before yield] 'sum_demo' started
  [inside with]  user code running...
  [after yield]  'sum_demo' finished — 0.000962s
t['elapsed'] = 0.000962s  |  sum = 4999950000


In [6]:
# =============================================================================
# PAUSE / RESUME — ASCII DIAGRAM (CODE COMMENT)
# =============================================================================

# simple_timer("sum_demo") called
# │
# │  container = {}
# │  start = perf_counter()
# │  print("[before yield] ...")
# │  yield container  ──────────────────────────────→  GENERATOR PAUSES
# │                                                    t = container  (same object)
# │
# │                       ┌── [with block body runs] ──┐
# │                       │  print("[inside with] ...")  │
# │                       │  total = sum(range(...))    │
# │                       └────────────────────────────┘
# │
# │  ←─────────────────────────────────────────────────  GENERATOR RESUMES
# │  elapsed = perf_counter() - start
# │  container["elapsed"] = elapsed    ← caller's `t` sees this immediately
# │  print("[after yield] ...")
# └─ done

print(f"Diagram above: t and container are the same dict at id {id(t):#x}")

Diagram above: t and container are the same dict at id 0x10d073340


---

## 3. Execution Flow — Step by Step

**Key insight: `yield` is a pause button.**

A normal function runs start-to-finish. A generator function **pauses** at
`yield` and **resumes** when `next()` is called again. `@contextmanager`
exploits this:

```
next() call #1  →  runs to yield   →  acts as __enter__
next() call #2  →  resumes to end  →  acts as __exit__
```

The `with` statement orchestrates both `next()` calls automatically.

### ASCII Execution Timeline

```
with traced_timer("demo") as t:
│
│  ①  generator created, next() called
│  ②  container = {}  /  start = perf_counter()
│  ③  yield container  ──────────────→  PAUSE
│                                       t = container  (the `as` target)
│
│  ④  t is {}  (empty — elapsed not yet computed)
│  ⑤  [user code: _ = sum(range(50_000))]
│
│  next() called again on generator
│  ⑥  generator RESUMES after yield
│  ⑦  elapsed computed; container["elapsed"] = elapsed
│  ⑧  generator ends → __exit__ complete
│
⑨  t = {"elapsed": ...}
⑩  t["elapsed"]  readable outside the block
```

In [7]:
# =============================================================================
# TRACED TIMER — PRINTS AT EVERY LIFECYCLE PHASE
# =============================================================================

@contextmanager
def traced_timer(label: str) -> Generator[dict[str, Any], None, None]:
    """A timer that announces every phase of its lifecycle.

    The numbered prints fire in the order shown in the ASCII timeline above,
    proving the yield pause-resume mechanism with real runtime output.

    Parameters
    ----------
    label : str
        Name shown at each execution step.

    Yields
    ------
    dict[str, Any]
        Empty on entry; ``"elapsed"`` key set after the block.
    """
    container: dict[str, Any] = {}

    print(f"  ① generator starts  label='{label}'")
    print(f"  ② before yield      recording start time")
    start = time.perf_counter()

    # Everything above ↑ runs as __enter__
    print(f"  ③ yield container   PAUSING — sending {{}} to caller")
    yield container

    # Everything below ↓ runs as __exit__ when the with block ends
    print(f"  ⑥ generator RESUMES block has finished")
    elapsed = time.perf_counter() - start
    container["elapsed"] = elapsed      # mutate the shared dict
    print(f"  ⑦ after yield       elapsed={elapsed:.6f}s stored in container")
    print(f"  ⑧ generator ends    __exit__ complete")

In [8]:
# =============================================================================
# RUNNING THE TRACED TIMER
# =============================================================================

with traced_timer("demo") as t:
    print(f"  ④ inside `with`     t is currently: {t}")   # {} — elapsed not yet set
    print(f"  ⑤ running work...")
    _ = sum(range(50_000))

print(f"  ⑨ after `with`      t is now: {t}")
print(f"  ⑩ t['elapsed']      = {t['elapsed']:.6f}s")

  ① generator starts  label='demo'
  ② before yield      recording start time
  ③ yield container   PAUSING — sending {} to caller
  ④ inside `with`     t is currently: {}
  ⑤ running work...
  ⑥ generator RESUMES block has finished
  ⑦ after yield       elapsed=0.000418s stored in container
  ⑧ generator ends    __exit__ complete
  ⑨ after `with`      t is now: {'elapsed': 0.0004181251861155033}
  ⑩ t['elapsed']      = 0.000418s


---

## 4. Why Yield a Mutable Container?

There is a timing problem: we `yield` **before** the timed block runs, but
we need to deliver the elapsed time **after** it finishes.

We cannot yield a `float` — floats are immutable. Reassigning `elapsed` after
the yield creates a *new* float object at a new memory address; the caller's
variable still points to the original `0.0`.

The fix: yield a **mutable container** (`dict` or `list`). Both caller and
generator hold a reference to the **same object in memory**, so any mutation
the generator makes after `yield` is immediately visible to the caller.

**Mental model — The Mailbox Pattern:**

```
yield container  →  You hand the caller an EMPTY MAILBOX (same address)
                    Caller holds a reference to that mailbox
                    User code runs
                    You put a LETTER in the mailbox (post-yield mutation)
                    Caller reads the letter — it is there, same box
```

**Python reference semantics:**

```
container = {}             # dict lives at memory address 0xABC
yield container            # caller's `t` → 0xABC  (same object)
container["elapsed"] = x   # writes INTO 0xABC — caller sees it immediately

# Why float fails:
elapsed = 0.0              # float at address 0xDEF
yield elapsed              # caller receives 0.0 — a copy of the VALUE
elapsed = 1.23             # NEW float at 0xFFF — caller's variable unchanged
```

In [9]:
# =============================================================================
# BROKEN — YIELDING AN IMMUTABLE FLOAT
# =============================================================================

@contextmanager
def broken_timer() -> Generator[float, None, None]:
    """BROKEN: yields a float.

    Post-yield reassignment creates a new float; the caller's variable
    still holds the original 0.0.
    """
    elapsed = 0.0
    start = time.perf_counter()
    yield elapsed                           # caller gets the VALUE 0.0
    elapsed = time.perf_counter() - start   # new float object — caller never sees it


with broken_timer() as t:
    _ = sum(range(50_000))

print(f"broken  → t = {t}  (still 0.0; the update never reached the caller)")

broken  → t = 0.0  (still 0.0; the update never reached the caller)


In [10]:
# =============================================================================
# CORRECT — YIELDING A MUTABLE DICT
# =============================================================================

@contextmanager
def working_timer() -> Generator[dict[str, Any], None, None]:
    """Correct: yields a mutable dict.

    Both generator and caller hold a reference to the SAME dict object.
    Any mutation after yield is immediately visible to the caller.

    Yields
    ------
    dict[str, Any]
        Empty on entry; ``"elapsed"`` (float) added after the block.
    """
    container: dict[str, Any] = {}         # one object — two references
    start = time.perf_counter()
    yield container                         # caller's `t` → this exact dict
    elapsed = time.perf_counter() - start
    container["elapsed"] = elapsed          # mutates the shared object


with working_timer() as t:
    _ = sum(range(50_000))

print(f"working → t = {t}")
print(f"         t['elapsed'] = {t['elapsed']:.6f}s")

working → t = {'elapsed': 0.0003977091982960701}
         t['elapsed'] = 0.000398s


In [11]:
# =============================================================================
# ALSO CORRECT — YIELDING A MUTABLE LIST
# =============================================================================

@contextmanager
def list_timer() -> Generator[list[Any], None, None]:
    """Alternative: yields a mutable list.

    Same principle as the dict — caller and generator share one list object.
    Less readable than a dict (no named key) but identical in mechanics.

    Yields
    ------
    list[Any]
        Empty on entry; ``list[0]`` is elapsed seconds after the block.
    """
    container: list[Any] = []
    start = time.perf_counter()
    yield container                         # caller's `t` → this same list
    elapsed = time.perf_counter() - start
    container.append(elapsed)               # append mutates the shared list


with list_timer() as t:
    _ = sum(range(50_000))

print(f"list    → t = {t}")
print(f"         t[0] = {t[0]:.6f}s")

list    → t = [0.0006432500667870045]
         t[0] = 0.000643s


---

## 5. Decorators Recap — Two-Layer vs Three-Layer

Before composing context managers with decorators, we need the two patterns precise.

**Two-layer — decorator without parameters:**

```python
def my_decorator(func):            # Layer 1: receives the FUNCTION
    @wraps(func)
    def wrapper(*args, **kwargs):  # Layer 2: receives CALL ARGUMENTS
        before()
        result = func(*args, **kwargs)
        after()
        return result
    return wrapper

@my_decorator          # no parentheses
def greet(name): ...
# Python executes: greet = my_decorator(greet)
```

**Three-layer — decorator with parameters:**

```python
def my_decorator(label):              # Layer 1: receives the PARAMETER
    def decorator(func):              # Layer 2: receives the FUNCTION
        @wraps(func)
        def wrapper(*args, **kwargs): # Layer 3: receives CALL ARGUMENTS
            print(f"[{label}] before")
            result = func(*args, **kwargs)
            print(f"[{label}] after")
            return result
        return wrapper
    return decorator

@my_decorator("greet")   # parentheses with argument
def greet(name): ...
# Python executes: greet = my_decorator("greet")(greet)
#                          ↑ returns decorator    ↑ called with the function
```

> **Rule of thumb:** `@deco` (no parens) → two-layer.
> `@deco(arg)` (parens) → three-layer; the outer call returns the decorator.

In [12]:
# =============================================================================
# TWO-LAYER DECORATOR (NO PARAMETERS)
# =============================================================================
from functools import wraps
from typing import Callable


def log_call(func: Callable) -> Callable:
    """Two-layer decorator — logs entry, return value, and preserves metadata.

    Parameters
    ----------
    func : Callable
        The function being decorated.

    Returns
    -------
    Callable
        Wrapped function with identical signature (preserved by @wraps).
    """
    @wraps(func)                           # copies __name__, __doc__, __annotations__
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        print(f"  → calling {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"  ← {func.__name__} returned {result!r}")
        return result
    return wrapper


@log_call                                  # no parens — two-layer
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


print(f"add.__name__ = {add.__name__!r}   ← @wraps preserved it")
result = add(3, 4)
print(f"final result : {result}")

add.__name__ = 'add'   ← @wraps preserved it
  → calling add(3, 4)
  ← add returned 7
final result : 7


In [13]:
# =============================================================================
# THREE-LAYER DECORATOR (WITH PARAMETERS)
# =============================================================================

def log_call_with_label(label: str) -> Callable:
    """Three-layer decorator — wraps calls with a custom label tag.

    Parameters
    ----------
    label : str
        Tag shown in log output, e.g. ``"MATH"`` or ``"DB"``.

    Returns
    -------
    Callable
        A standard two-layer decorator waiting for a function.
    """
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            print(f"  [{label}] → calling {func.__name__}{args}")
            result = func(*args, **kwargs)
            print(f"  [{label}] ← {func.__name__} returned {result!r}")
            return result
        return wrapper
    return decorator


@log_call_with_label("MATH")               # parens with arg — three-layer
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


print(f"multiply.__name__ = {multiply.__name__!r}   ← @wraps preserved it")
result = multiply(3, 4)
print(f"final result      : {result}")

multiply.__name__ = 'multiply'   ← @wraps preserved it
  [MATH] → calling multiply(3, 4)
  [MATH] ← multiply returned 12
final result      : 12


---

## 6. Wrapping a Context Manager in a Decorator

This is the **key production pattern**: build timing logic *once* in a context
manager, then reuse it inside a decorator — DRY, composable, independently testable.

- Use the **context manager** directly for **block-level** timing of arbitrary code.
- Apply the **decorator** for **function-level** timing without touching the body.

### How `timed_decorator` wraps `timer_cm`

```
@timed_decorator("load")
def load_dict(path): ...

When load_dict(path) is called:

┌─ wrapper(path) entered
│
│  ┌─ with timer_cm("load") as t:     ← opens the context manager
│  │   start = perf_counter()
│  │   yield container  ──→ PAUSE
│  │
│  │   result = load_dict(path)        ← original function runs here
│  │
│  └─ block ends → generator resumes
│      container["elapsed"] = elapsed
│
│  wrapper.benchmark = t["elapsed"]    ← stored on the function object
│  return result                       ← original return value unchanged
└─ done
```

In [14]:
# =============================================================================
# THE CONTEXT MANAGER (REUSABLE FOUNDATION)
# =============================================================================

@contextmanager
def timer_cm(label: str) -> Generator[dict[str, Any], None, None]:
    """Time a code block; store elapsed seconds in a shared dict.

    This is the reusable primitive. The decorator below calls it —
    it never duplicates the timing logic.

    Parameters
    ----------
    label : str
        Human-readable name for this timing event.

    Yields
    ------
    dict[str, Any]
        Empty on entry; ``"elapsed"`` (float) set after the block.
    """
    container: dict[str, Any] = {}
    print(f"  [timer_cm] start '{label}'")
    start = time.perf_counter()
    yield container                         # user code runs here
    elapsed = time.perf_counter() - start
    container["elapsed"] = elapsed
    print(f"  [timer_cm] end   '{label}' — {elapsed:.6f}s")

In [15]:
# =============================================================================
# THE DECORATOR THAT WRAPS timer_cm
# =============================================================================

def timed_decorator(label: str) -> Callable:
    """Three-layer decorator that times a function via timer_cm.

    The decorated function's return value is unchanged.
    After each call, the elapsed time is stored on ``func.benchmark``.

    Parameters
    ----------
    label : str
        Passed to timer_cm for log output.

    Returns
    -------
    Callable
        A decorator that wraps any callable with timing via timer_cm.

    Examples
    --------
    >>> @timed_decorator("compute")
    ... def heavy(n): return sum(range(n))
    >>> heavy(100_000)
    4999950000
    >>> heavy.benchmark   # seconds
    """
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            # KEY PATTERN — reuse the context manager inside the decorator
            with timer_cm(label) as t:
                result = func(*args, **kwargs)   # original function unchanged

            wrapper.benchmark = t["elapsed"]     # type: ignore[attr-defined]
            return result                         # return value unchanged

        wrapper.benchmark = None                 # type: ignore[attr-defined]
        return wrapper
    return decorator

In [16]:
# =============================================================================
# BLOCK-LEVEL USAGE (context manager directly)
# =============================================================================

# Use timer_cm directly when you need to time an arbitrary block of code
with timer_cm("sum — block level") as t:
    total = sum(range(100_000))

print(f"elapsed = {t['elapsed']:.6f}s  |  sum = {total}")

  [timer_cm] start 'sum — block level'
  [timer_cm] end   'sum — block level' — 0.000835s
elapsed = 0.000835s  |  sum = 4999950000


In [17]:
# =============================================================================
# FUNCTION-LEVEL USAGE (decorator)
# =============================================================================

@timed_decorator("compute_sum")
def compute_sum(n: int) -> int:
    """Return sum(range(n)).

    Parameters
    ----------
    n : int
        Upper bound (exclusive).

    Returns
    -------
    int
        Sum of integers 0 .. n-1.
    """
    return sum(range(n))


result = compute_sum(100_000)
print(f"return value          = {result}")            # unchanged by decorator
print(f"compute_sum.benchmark = {compute_sum.benchmark:.6f}s")   # type: ignore[attr-defined]
print(f"compute_sum.__name__  = {compute_sum.__name__!r}")        # preserved by @wraps

  [timer_cm] start 'compute_sum'
  [timer_cm] end   'compute_sum' — 0.001054s
return value          = 4999950000
compute_sum.benchmark = 0.001054s
compute_sum.__name__  = 'compute_sum'


---

## 7. Common Patterns and Real-World Usage

Four patterns that appear throughout the Python ecosystem — from stdlib
to the ML and LLMOps frameworks on a GenAI roadmap.

| Pattern | What it does | Real-world examples |
|---|---|---|
| **Temporary state change** | Set → yield → restore | log level, Streamlit state, config override |
| **Exception-safe cleanup** | `try / yield / finally` | DB connections, file handles, sockets |
| **Accumulating timer** | Add elapsed to running total per call | Speller `check()` hot loop |
| **Yield `None`** | Setup/teardown, no value for caller | `torch.no_grad()`, `mlflow.start_run()` |

**Yield `None`** is used when the caller doesn't need a value — just the
side-effect of the setup/teardown. Use bare `yield` and omit `as X`:

```python
@contextmanager
def side_effect_cm():
    setup()
    yield               # bare yield — caller writes `with side_effect_cm():`
    teardown()
```

In [18]:
# =============================================================================
# PATTERN 1 — TEMPORARY STATE CHANGE
# =============================================================================
import logging

# Temporarily override a logger's level, then restore the original.
# Real-world: test fixtures, Streamlit debug mode, DataVault LLM call tracing.


@contextmanager
def temporary_log_level(
    logger: logging.Logger,
    level: int,
) -> Generator[None, None, None]:
    """Temporarily change a logger's level; restore the original on exit.

    Parameters
    ----------
    logger : logging.Logger
        Logger whose level will be overridden.
    level : int
        Target level (e.g. ``logging.DEBUG = 10``).

    Yields
    ------
    None
        Caller uses the block for the side effect; no ``as X`` needed.
    """
    original = logger.level
    logger.setLevel(level)       # mutate
    yield                        # bare yield — no value to pass
    logger.setLevel(original)    # restore — always runs


demo_logger = logging.getLogger("demo")
demo_logger.setLevel(logging.WARNING)

print(f"before : logger.level = {demo_logger.level}  (WARNING=30)")
with temporary_log_level(demo_logger, logging.DEBUG):
    print(f"during : logger.level = {demo_logger.level}  (DEBUG=10)")
print(f"after  : logger.level = {demo_logger.level}  (WARNING=30 restored)")

before : logger.level = 30  (WARNING=30)
during : logger.level = 10  (DEBUG=10)
after  : logger.level = 30  (WARNING=30 restored)


In [19]:
# =============================================================================
# PATTERN 2 — EXCEPTION-SAFE CLEANUP
# =============================================================================

# try/finally guarantees cleanup even when the body raises.
# Real-world: psycopg2.connect(), boto3.resource(), chromadb.Client(),
#             mlflow.start_run(), LangGraph state managers.


@contextmanager
def managed_connection(name: str) -> Generator[dict[str, Any], None, None]:
    """Simulate a DB connection with guaranteed cleanup on exceptions.

    Parameters
    ----------
    name : str
        Logical name of the connection shown in log output.

    Yields
    ------
    dict[str, Any]
        Fake connection object with ``"name"`` and ``"open"`` keys.
    """
    conn: dict[str, Any] = {"name": name, "open": True}
    print(f"  [connect]  opened  '{name}'")
    try:
        yield conn               # give caller the connection
    except Exception as exc:
        print(f"  [error]    '{name}' caught {type(exc).__name__}: {exc}")
        raise                    # re-raise — caller still sees the error
    finally:
        # finally runs ALWAYS — happy path and exception path alike
        conn["open"] = False
        print(f"  [cleanup]  closed  '{name}'  (guaranteed)")


# Happy path
print("--- happy path ---")
with managed_connection("prod-db") as conn:
    pass   # use the connection
print(f"conn['open'] = {conn['open']}\n")

# Error path — cleanup still fires
print("--- error path ---")
try:
    with managed_connection("bad-db") as conn:
        raise ValueError("simulated query error")
except ValueError:
    pass
print(f"conn['open'] = {conn['open']}  ← closed even after exception")

--- happy path ---
  [connect]  opened  'prod-db'
  [cleanup]  closed  'prod-db'  (guaranteed)
conn['open'] = False

--- error path ---
  [connect]  opened  'bad-db'
  [error]    'bad-db' caught ValueError: simulated query error
  [cleanup]  closed  'bad-db'  (guaranteed)
conn['open'] = False  ← closed even after exception


In [20]:
# =============================================================================
# PATTERN 3 — ACCUMULATING TIMER (LOOP USAGE)
# =============================================================================

# Add elapsed time to a running total across many loop iterations.
# This is exactly how Speller times check(): thousands of calls,
# one cumulative total — not one timer per call.


@contextmanager
def accumulating_timer(
    label: str,
    accumulator: dict[str, float],
) -> Generator[None, None, None]:
    """Add elapsed time for this iteration to a running total.

    Parameters
    ----------
    label : str
        Key inside ``accumulator`` where time is summed.
    accumulator : dict[str, float]
        Shared dict that persists across calls; values are added cumulatively.

    Yields
    ------
    None
        Caller uses the block for the side effect of timing.
    """
    start = time.perf_counter()
    yield                                        # time this single iteration
    accumulator[label] = (
        accumulator.get(label, 0.0) + (time.perf_counter() - start)
    )


WORD_SET = {"hello", "world", "python"}          # simulated hash-table dictionary
words = ["hello", "world", "python", "speller", "benchmark", "context", "manager"]
totals: dict[str, float] = {}

for word in words:
    with accumulating_timer("check", totals):
        _ = word.lower() in WORD_SET             # O(1) lookup — what Speller does

print(f"checked {len(words)} words")
print(f"total check() time = {totals['check']:.6f}s  (cumulative across all calls)")

checked 7 words
total check() time = 0.000004s  (cumulative across all calls)


---

## 8. Gotchas and Pitfalls

The most common mistakes when using `@contextmanager` and decorator composition.

| # | Pitfall | Fix |
|---|---|---|
| 1 | Multiple `yield`s in `@contextmanager` | Exactly one `yield` |
| 2 | Missing `@wraps(func)` | Always add `@wraps` |
| 3 | Post-`yield` cleanup skipped by exceptions | Wrap in `try / finally` |
| 4 | Dynamic function attributes confuse mypy | `# type: ignore[attr-defined]` |

In [21]:
# =============================================================================
# GOTCHA 1 — MULTIPLE YIELDS (RuntimeError)
# =============================================================================

# @contextmanager generators must yield EXACTLY ONCE.
# A second yield raises RuntimeError: "generator didn't stop".

@contextmanager
def double_yield_demo() -> Generator[str, None, None]:
    """BROKEN: two yields — only to show the error safely."""
    yield "first"
    yield "second"   # ← will cause RuntimeError when the with block ends


try:
    with double_yield_demo() as v:
        print(f"  got value: {v!r}")
        # block ends here → contextmanager calls next() → hits second yield
except RuntimeError as exc:
    print(f"  RuntimeError: {exc}")

  got value: 'first'
  RuntimeError: generator didn't stop


In [22]:
# =============================================================================
# GOTCHA 2 — MISSING @WRAPS
# =============================================================================

# Without @wraps, the decorator overwrites __name__, __doc__, __annotations__.
# This breaks introspection, help(), sphinx docs, and pytest output.

def bad_deco(func: Callable) -> Callable:
    def wrapper(*a: Any, **kw: Any) -> Any:   # ← no @wraps
        return func(*a, **kw)
    return wrapper


def good_deco(func: Callable) -> Callable:
    @wraps(func)                               # ← copies all metadata
    def wrapper(*a: Any, **kw: Any) -> Any:
        return func(*a, **kw)
    return wrapper


@bad_deco
def my_func_bad() -> str:
    """Important docstring."""
    return "hello"


@good_deco
def my_func_good() -> str:
    """Important docstring."""
    return "hello"


print(f"without @wraps : __name__={my_func_bad.__name__!r:12}  __doc__={my_func_bad.__doc__!r}")
print(f"with @wraps    : __name__={my_func_good.__name__!r:12}  __doc__={my_func_good.__doc__!r}")

without @wraps : __name__='wrapper'     __doc__=None
with @wraps    : __name__='my_func_good'  __doc__='Important docstring.'


In [23]:
# =============================================================================
# GOTCHA 3 — CLEANUP SKIPPED WITHOUT try / finally
# =============================================================================

# Code written after yield is NOT in a finally block — it only runs
# if no exception fires. Add try/finally to guarantee cleanup.

# ❌  WRONG — release() never runs if an exception fires in the with block:
#
# @contextmanager
# def bad_cleanup():
#     resource = acquire()
#     yield resource
#     release(resource)       # ← skipped on exception!
#
# ✅  CORRECT — finally always runs:
#
# @contextmanager
# def good_cleanup():
#     resource = acquire()
#     try:
#         yield resource
#     finally:
#         release(resource)   # ← runs on both happy path and exception path

# Demonstrate the difference:
released_bad: bool = False
released_good: bool = False


@contextmanager
def bad_cleanup() -> Generator[None, None, None]:
    global released_bad
    yield
    released_bad = True        # ← only reached if no exception


@contextmanager
def good_cleanup() -> Generator[None, None, None]:
    global released_good
    try:
        yield
    finally:
        released_good = True   # ← always reached


try:
    with bad_cleanup():
        raise RuntimeError("simulated failure")
except RuntimeError:
    pass

try:
    with good_cleanup():
        raise RuntimeError("simulated failure")
except RuntimeError:
    pass

print(f"bad_cleanup  released = {released_bad}   ← cleanup was skipped!")
print(f"good_cleanup released = {released_good}  ← cleanup ran despite exception")

bad_cleanup  released = False   ← cleanup was skipped!
good_cleanup released = True  ← cleanup ran despite exception


In [24]:
# =============================================================================
# GOTCHA 4 — DYNAMIC ATTRIBUTES AND MYPY
# =============================================================================

# Setting wrapper.benchmark = value makes mypy complain:
#   error: "Callable[..., Any]" has no attribute "benchmark"  [attr-defined]
# because Callable has no .benchmark declared in its type stub.

# Three options:

# 1. Targeted suppress (most common in practice):
#    wrapper.benchmark = value  # type: ignore[attr-defined]

# 2. Cast to Any (avoids the ignore comment):
from typing import cast
wrapper_any: Any = compute_sum
print(f"benchmark via cast : {wrapper_any.benchmark:.6f}s")

# 3. Protocol (strict, most correct for library code):
#
# from typing import Protocol
# class TimedCallable(Protocol):
#     benchmark: float | None
#     def __call__(self, n: int) -> int: ...
#
# fn: TimedCallable = compute_sum   # mypy is satisfied, no ignore needed

benchmark via cast : 0.001054s


---

## Quick Reference

```python
# ─── @contextmanager basic form ──────────────────────────────────────────
from contextlib import contextmanager
from typing import Generator

@contextmanager
def my_cm(arg: str) -> Generator[dict, None, None]:
    container: dict = {}
    setup(arg)             # __enter__
    yield container        # value → `as X`; generator PAUSES here
    cleanup()              # __exit__  (only reached on no exception!)


# ─── exception-safe form ─────────────────────────────────────────────────
@contextmanager
def safe_cm(arg: str) -> Generator[Resource, None, None]:
    resource = acquire(arg)
    try:
        yield resource
    finally:
        release(resource)  # ALWAYS runs — happy path or exception


# ─── yield None (no value needed) ────────────────────────────────────────
@contextmanager
def side_effect_cm() -> Generator[None, None, None]:
    setup()
    yield               # bare yield — caller writes `with side_effect_cm():`
    teardown()


# ─── two-layer decorator (no parameters) ─────────────────────────────────
def deco(func: Callable) -> Callable:
    @wraps(func)
    def wrapper(*args, **kwargs):
        before()
        result = func(*args, **kwargs)
        after()
        return result
    return wrapper

@deco
def my_func(): ...


# ─── three-layer decorator (with parameters) ──────────────────────────────
def deco(param: str) -> Callable:
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs):
            before(param)
            result = func(*args, **kwargs)
            after(param)
            return result
        return wrapper
    return decorator

@deco("label")
def my_func(): ...


# ─── decorator wrapping a context manager ─────────────────────────────────
def timed(name: str) -> Callable:
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs):
            with timer_cm(name) as t:
                result = func(*args, **kwargs)
            wrapper.benchmark = t["elapsed"]  # type: ignore[attr-defined]
            return result
        wrapper.benchmark = None              # type: ignore[attr-defined]
        return wrapper
    return decorator


# ─── accumulating timer ───────────────────────────────────────────────────
totals: dict[str, float] = {}
for item in items:
    with accumulating_timer("op", totals):
        do_work(item)
print(totals["op"])   # cumulative seconds across all calls


# ─── key rules ────────────────────────────────────────────────────────────
# • @contextmanager generators must yield EXACTLY once
# • Yield a MUTABLE container (dict/list) to share data post-yield
# • Always @wraps(func) in decorators — preserves __name__, __doc__
# • Use try/finally for guaranteed cleanup (bare post-yield code is NOT safe)
# • time.perf_counter() is monotonic — always prefer it over time.time()
# • Return False from __exit__ to let exceptions propagate (safe default)
# • Dynamic attributes: wrapper.attr = v  # type: ignore[attr-defined]
```